# HotelIQ – Data Preprocessing

## Objective

The objective of this notebook is to prepare the selected features for machine learning by performing data splitting, categorical encoding, and numerical feature scaling.

### Steps Performed
- Load the cleaned hotel booking dataset.
- Separate independent and dependent variables.
- Split the dataset into training and testing sets.
- Encode categorical features.
- Scale numerical features.
- Prepare a ColumnTransformer for model preprocessing.

In [ ]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/hotel_bookings_cleaned.csv')

print("Dataset Shape: ",df.shape)

Dataset Shape:  (87395, 31)


In [3]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,-1.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,-1.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,-1.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240.0,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87395 entries, 0 to 87394
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87395 non-null  str    
 1   is_canceled                     87395 non-null  int64  
 2   lead_time                       87395 non-null  int64  
 3   arrival_date_year               87395 non-null  int64  
 4   arrival_date_month              87395 non-null  str    
 5   arrival_date_week_number        87395 non-null  int64  
 6   arrival_date_day_of_month       87395 non-null  int64  
 7   stays_in_weekend_nights         87395 non-null  int64  
 8   stays_in_week_nights            87395 non-null  int64  
 9   adults                          87395 non-null  int64  
 10  children                        87395 non-null  float64
 11  babies                          87395 non-null  int64  
 12  meal                            87395 non-n

### Separating Features and Target

The target variable is `is_canceled`, which indicates whether a hotel booking was cancelled.

- **X:** Independent variables used to predict cancellation.
- **y:** Dependent variable that we want to predict.

Columns that could cause data leakage or are unavailable at the time of booking will be excluded.

In [6]:
# columns to exclude
drop_columns = ['is_canceled', 'reservation_status',
                'reservation_status_date', 'booking_changes',
                'assigned_room_type', 'days_in_waiting_list']

# Separate features and target
X = df.drop(columns=drop_columns)
y = df['is_canceled']

print("Features Shape: ",X.shape)
print("Target Shape: ", y.shape)

Features Shape:  (87395, 25)
Target Shape:  (87395,)


In [9]:
# Categorical features
categorical_features = ['hotel', 'arrival_date_month', 'meal', 
                        'country', 'market_segment', 'distribution_channel',
                        'reserved_room_type', 'deposit_type', 'customer_type', 'agent']

# Numerical features
numerical_features = [col for col in X.columns
                      if col not in categorical_features]

print("Numerical Features:", len(numerical_features))
print("Categorical Features:", len(categorical_features))

print("\nCategorical Columns:")
print(categorical_features)

print("\nNumerical Columns:")
print(numerical_features)

Numerical Features: 15
Categorical Features: 10

Categorical Columns:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type', 'agent']

Numerical Columns:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


### Train-Test Split

The dataset is divided into training and testing sets.

- 80% of the data is used for training.
- 20% of the data is used for testing.
- Stratified sampling is applied to maintain the target class distribution in both sets.

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=49, stratify=y)

print("Training Features: ", X_train.shape)

print("Testing Features: ", X_test.shape)

print("\nTraining Target Distribution")
print(y_train.value_counts(normalize=True)*100)

print("\nTesting Target Distribution")
print(y_test.value_counts(normalize=True)*100)

Training Features:  (69916, 25)
Testing Features:  (17479, 25)

Training Target Distribution
is_canceled
0    72.509869
1    27.490131
Name: proportion, dtype: float64

Testing Target Distribution
is_canceled
0    72.509869
1    27.490131
Name: proportion, dtype: float64


### Feature Transformation

Machine learning algorithms require numerical inputs.

Therefore, the following transformations are applied:

- Numerical features are standardized using StandardScaler.
- Categorical features are converted into numerical representations using OneHotEncoder.
- ColumnTransformer combines both transformations into a single preprocessing pipeline.

The transformations will be fitted only on the training data to prevent data leakage.

In [21]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [25]:
# create preprocessing transformer

preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numerical_features),
                                               ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

print("Preprocessor created successfully!")

Preprocessor created successfully!


### Applying Preprocessing Transformations

The preprocessor is fitted on the training dataset and transforms its numerical and categorical features.

- `fit_transform()` learns the scaling parameters and categorical encoding from training data.
- The learned transformations will later be applied to the testing dataset using `transform()`.

This ensures that information from the testing dataset is not used during preprocessing.

In [26]:
# Fit and transform training data
X_train_transformed = preprocessor.fit_transform(X_train)

print("Training Data Shape Before Transformation:", X_train.shape)
print("Training Data Shape After Transformation:", X_train_transformed.shape)

Training Data Shape Before Transformation: (69916, 25)
Training Data Shape After Transformation: (69916, 555)


In [27]:
# Transform testing data
X_test_transformed = preprocessor.transform(X_test)

print("Testing Data Shape Before Transformation:", X_test.shape)
print("Testing Data Shape After Transformation:", X_test_transformed.shape)

Testing Data Shape Before Transformation: (17479, 25)
Testing Data Shape After Transformation: (17479, 555)


In [29]:
from scipy import sparse

# Check missing and infinite values in training data
if sparse.issparse(X_train_transformed):
    train_data = X_train_transformed.data
else:
    train_data = X_train_transformed

# Check missing and infinite values in testing data
if sparse.issparse(X_test_transformed):
    test_data = X_test_transformed.data
else:
    test_data = X_test_transformed

print("Missing values in training data:", np.isnan(train_data).sum())
print("Missing values in testing data:", np.isnan(test_data).sum())

print("Infinite values in training data:", np.isinf(train_data).sum())
print("Infinite values in testing data:", np.isinf(test_data).sum())

Missing values in training data: 0
Missing values in testing data: 0
Infinite values in training data: 0
Infinite values in testing data: 0


## Preprocessing Summary

The following preprocessing steps were completed:

1. Loaded the cleaned hotel booking dataset.
2. Separated independent features (X) and target variable (y).
3. Identified 15 numerical and 10 categorical features.
4. Split the dataset into training (80%) and testing (20%) sets using stratified sampling.
5. Applied StandardScaler to numerical features.
6. Applied OneHotEncoder to categorical features.
7. Used ColumnTransformer to combine both transformations.
8. Fitted the preprocessor on training data and applied the same transformations to testing data.
9. Verified that both transformed datasets contain no missing or infinite values.

### Final Dataset Shapes

| Dataset | Shape |
|---|---|
| Training Data | (69916, 555) |
| Testing Data | (17479, 555) |

The datasets are now preprocessed and ready for machine learning model development.